In [2]:
!pip install -q transformers accelerate bitsandbytes peft datasets

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [4]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("✅ Model loaded in 4-bit")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✅ Model loaded in 4-bit


In [5]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()
print("✅ LoRA applied")

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079
✅ LoRA applied


In [6]:
from datasets import load_dataset
from google.colab import files
import os

# -------------------------------
# STEP 1: Upload dataset files
# -------------------------------
print("📂 Upload train.jsonl and val.jsonl")
uploaded = files.upload()

# -------------------------------
# STEP 2: Verify files
# -------------------------------
print("📁 Current directory files:", os.listdir())

# -------------------------------
# STEP 3: Load dataset
# -------------------------------
dataset = load_dataset("json", data_files={
    "train": "train.jsonl",
    "validation": "val.jsonl"
})

print("✅ Dataset loaded")

# -------------------------------
# STEP 4: Tokenization
# -------------------------------
def tokenize_function(example):
    prompt = f"""### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['output']}"""

    tokens = tokenizer(
        prompt,
        truncation=True,
        padding="max_length",
        max_length=256
    )

    # 🔥 Important for training
    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

tokenized_datasets = dataset.map(tokenize_function, remove_columns=dataset["train"].column_names)

print("✅ Dataset tokenized and ready")

📂 Upload train.jsonl and val.jsonl


Saving train.jsonl to train.jsonl
Saving val.jsonl to val.jsonl
📁 Current directory files: ['.config', 'train.jsonl', 'val.jsonl', 'sample_data']


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

✅ Dataset loaded


Map:   0%|          | 0/1080 [00:00<?, ? examples/s]

Map:   0%|          | 0/120 [00:00<?, ? examples/s]

✅ Dataset tokenized and ready


In [7]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=1,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=100,
    save_total_limit=2,
    report_to="none"
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator
)

print("🚀 Training setup ready")

🚀 Training setup ready


In [8]:
trainer.train()

print("🎉 Training completed")

Step,Training Loss,Validation Loss
50,0.046530,0.053363
100,0.037726,0.035092
150,0.026273,0.035625
200,0.030611,0.031612
250,0.032913,0.034430
300,0.036013,0.031605
350,0.033517,0.032993
400,0.031625,0.034013
450,0.036083,0.031346
500,0.027121,0.031741


🎉 Training completed


In [9]:
from google.colab import drive
drive.mount('/content/drive')

save_path = "/content/drive/MyDrive/tinyllama-adapters"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("✅ Adapter saved successfully")

Mounted at /content/drive
✅ Adapter saved successfully


In [10]:
import os
print(os.listdir("/content/drive/MyDrive/tinyllama-adapters"))

['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json']
